[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/notebooks/survivorship_bias.ipynb)

# Valuein US Core Fundamentals — Survivorship-Bias-Free Data

Most financial databases quietly delete companies that went bankrupt,
were acquired, or were delisted. What's left is a graveyard of winners.
Backtest a momentum strategy on that universe and your Sharpe looks
great — because you never tested it on the companies that blew up.

We kept them. Every last one.

**What you'll learn:**
- How many inactive/delisted companies are in the dataset
- How to find former index members that departed (via `index_membership`)
- How to query their financials in the years before failure
- Why omitting these companies overstates backtested returns

---

In [ ]:
!pip install valuein-sdk -q

In [ ]:
try:
    import os

    from google.colab import userdata

    os.environ["VALUEIN_API_KEY"] = userdata.get("VALUEIN_API_KEY")
except ImportError:
    pass

In [ ]:
from valuein_sdk import ValueinClient

client = ValueinClient(
    tables=["entity", "security", "fact", "filing", "index_membership"]
)

## 1. The companies most providers deleted

How many entities in the dataset are inactive, delisted, or bankrupt?
Most providers silently remove these — we show exactly how large that hidden universe is.

In [ ]:
df = client.query("""
    SELECT
        status,
        count(*)                                              AS companies,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1)   AS pct_of_universe
    FROM entity
    GROUP BY status
    ORDER BY companies DESC
""")
print(df.to_string(index=False))

## 2. Former S&P 500 members that left the index

`index_membership` tracks every company that ever joined an index.
`end_date IS NOT NULL` means they left — acquired, delisted, or went bankrupt.
These are the companies data providers deleted.

In [ ]:
df = client.query("""
    SELECT
        e.name,
        e.status,
        s.symbol,
        im.start_date   AS joined_index,
        im.end_date     AS left_index
    FROM   index_membership im
    JOIN   security s ON s.id        = im.security_id
    JOIN   entity   e ON e.cik       = s.entity_id
    WHERE  im.index_name = 'S&P 500'
      AND  im.end_date IS NOT NULL
    QUALIFY row_number() OVER (PARTITION BY e.cik ORDER BY im.end_date DESC) = 1
    ORDER BY im.end_date DESC
    LIMIT 20
""")
print(df.to_string(index=False))

## 3. Revenue and net income for a former index member (pre-exit)

Pick the most recent S&P 500 departure with fact data, then show its income trend
in the years before it left the index. This is the data that disappears from
survivorship-biased databases.

In [ ]:
df = client.query("""
    WITH departed AS (
        SELECT
            e.cik,
            e.name,
            im.end_date
        FROM   index_membership im
        JOIN   security s ON s.id    = im.security_id
        JOIN   entity   e ON e.cik   = s.entity_id
        WHERE  im.index_name = 'S&P 500'
          AND  im.end_date IS NOT NULL
        ORDER  BY im.end_date DESC
        LIMIT  1
    )
    SELECT
        d.name,
        fa.fiscal_year,
        fa.standard_concept,
        round(fa.numeric_value / 1e9, 3) AS value_bn
    FROM   fact   fa
    JOIN   filing f ON fa.accession_id = f.accession_id
    JOIN   departed d ON fa.entity_id  = d.cik
    WHERE  fa.standard_concept IN ('Revenues', 'NetIncomeLoss')
      AND  f.form_type = '10-K'
      AND  fa.fiscal_period = 'FY'
    QUALIFY row_number() OVER (
        PARTITION BY fa.fiscal_year, fa.standard_concept
        ORDER BY fa.period_end DESC
    ) = 1
    ORDER  BY fa.fiscal_year DESC, fa.standard_concept
    LIMIT  20
""")
if not df.empty:
    company = df["name"].iloc[0]
    print(f"  Company: {company}")
    print(df[["fiscal_year", "standard_concept", "value_bn"]].to_string(index=False))
else:
    # Fallback: any inactive entity with facts
    df = client.query("""
        WITH target AS (
            SELECT e.cik, e.name
            FROM   entity e
            WHERE  e.status != 'ACTIVE'
              AND  e.sector IS NOT NULL
            LIMIT  1
        )
        SELECT
            t.name,
            fa.fiscal_year,
            fa.standard_concept,
            round(fa.numeric_value / 1e9, 3) AS value_bn
        FROM   fact   fa
        JOIN   filing f ON fa.accession_id = f.accession_id
        JOIN   target t ON fa.entity_id    = t.cik
        WHERE  fa.standard_concept IN ('Revenues', 'NetIncomeLoss')
          AND  f.form_type = '10-K'
          AND  fa.fiscal_period = 'FY'
        QUALIFY row_number() OVER (
            PARTITION BY fa.fiscal_year, fa.standard_concept
            ORDER BY fa.period_end DESC
        ) = 1
        ORDER  BY fa.fiscal_year DESC, fa.standard_concept
        LIMIT  20
    """)
    if not df.empty:
        company = df["name"].iloc[0]
        print(f"  Company: {company}")
        print(
            df[["fiscal_year", "standard_concept", "value_bn"]].to_string(index=False)
        )
    else:
        print("  (No inactive companies with fact data in this plan tier)")

## 4. Why it matters

Let's quantify the survivorship bias problem: how many companies have truly
ceased to exist, and how many S&P 500 departures are "true" survivorship cases
vs companies that were simply demoted to smaller indices?

In [ ]:
# 1. Total Universe Stats
total_stats = client.query("""
    SELECT 
        count(*) as total,
        count(*) FILTER (WHERE status != 'ACTIVE') as inactive
    FROM entity
""").iloc[0]

# 2. Index Exit Stats - Distinguished by Status
exit_stats = client.query("""
    SELECT 
        e.status,
        count(DISTINCT s.entity_id) AS n
    FROM index_membership im
    JOIN security s ON s.id = im.security_id
    JOIN entity e ON e.cik = s.entity_id
    WHERE im.index_name = 'S&P 500' 
      AND im.end_date IS NOT NULL
    GROUP BY e.status
""")

# Parse numbers for the summary
total_count = total_stats["total"]
dead_count = total_stats["inactive"]
pct_dead = round(100.0 * dead_count / total_count, 1)

departed_active = exit_stats[exit_stats["status"] == "ACTIVE"]["n"].sum()
departed_inactive = exit_stats[exit_stats["status"] != "ACTIVE"]["n"].sum()
total_departed = departed_active + departed_inactive

print("  DATASET OVERVIEW:")
print(f"  - Total Entities: {total_count:,}")
print(f"  - Truly Inactive/Dead: {dead_count:,} ({pct_dead}%)")
print("\n  S&P 500 SURVIVORSHIP:")
print(f"  - {total_departed:,} companies left the S&P 500 index.")
print(
    f"    \u2514\u2500 {departed_active:,} are still trading (likely demoted to MidCap/SmallCap)."
)
print(
    f"    \u2514\u2500 {departed_inactive:,} are the true 'Survivorship Bias' cases (delisted/bankrupt)."
)
print("\n  A backtest that ignores the 'Inactive' subset is fundamentally flawed,")
print("  as it ignores the 100% losses that occur when a company ceases to exist.")